# Interactive Map & Custom Routing

In this notebook, we reuse the exact distances, candidate geometric points, and road graph computed in Notebook 02 to instantly generate a custom optimized route. We will build an interactive UI (using `ipywidgets`) so you can select a specific subset of animals to visit today, and let the MILP solver and Folium instantly map the shortest path for exactly those enclosures.

In [1]:
import osmnx as ox
import numpy as np
import folium
import pickle
from collections import defaultdict
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
# Load precomputed data
with open('../data/processed/optimization_results.pkl', 'rb') as f:
    opt_data = pickle.load(f)

all_names = opt_data['names']
candidate_nodes = opt_data['candidate_nodes']
dist_between_nodes = opt_data['dist_between_nodes']
paths_between_nodes = opt_data['paths_between_nodes']
pt_to_lonlat = opt_data['pt_to_lonlat']
candidates_utm = opt_data['candidates_utm']
pt_to_node = opt_data['pt_to_node']

# We also need the original graph to fetch raw GPS coordinates for each topological node
G_metros = ox.load_graphml('../data/processed/cabarceno_graph.graphml')

print(f"Loaded interactive data with {len(all_names)} enclosures.")

Loaded interactive data with 31 enclosures.


## 1. Core Functions

In [3]:
def solve_custom_tsp(selected_names):
    """
    Takes a subset of active enclosures. Ensures 'Main Entrance' is present.
    Solves the Generalized TSP via OR-Tools mapping multiple parking candidates.
    Returns the optimal list of topological candidate indices, and the sorted name order.
    """
    if 'Main Entrance' not in selected_names:
        selected_names = ['Main Entrance'] + list(selected_names)
        
    flat_nodes = []
    enclosure_to_indices_map = []
    global_idx = 0
    
    # 1. Prepare flat matrix indexing
    for name in selected_names:
        r_indices = []
        for nd in candidate_nodes[name]:
            flat_nodes.append(nd)
            r_indices.append(global_idx)
            global_idx += 1
        enclosure_to_indices_map.append(r_indices)

    N = len(flat_nodes)
    dm = np.zeros((N, N), dtype=int)
    for i, u in enumerate(flat_nodes):
        for j, v in enumerate(flat_nodes):
            if i != j:
                dm[i][j] = dist_between_nodes.get((u, v), 10**8)

    # 2. Configure PyWrapCP
    mgr = pywrapcp.RoutingIndexManager(N, 1, 0)
    mdl = pywrapcp.RoutingModel(mgr)

    def cb(from_idx, to_idx):
        return int(dm[mgr.IndexToNode(from_idx)][mgr.IndexToNode(to_idx)])

    cb_idx = mdl.RegisterTransitCallback(cb)
    mdl.SetArcCostEvaluatorOfAllVehicles(cb_idx)

    # 3. Add Disjunction Constraints (Exactly 1 parking per enclosure)
    non_visit_penalty = 10**8
    for r_indices in enclosure_to_indices_map[1:]:
        routes_nodes_cp = [mgr.NodeToIndex(i) for i in r_indices]
        mdl.AddDisjunction(routes_nodes_cp, non_visit_penalty)

    # 4. Parameters and Solve
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.seconds = 5  # Instant for subsets

    sol = mdl.SolveWithParameters(params)
    if not sol:
        return None, None, None
        
    best_distance = sol.ObjectiveValue()
    
    idx = mdl.Start(0)
    route_indices = []
    while not mdl.IsEnd(idx):
        node_idx = mgr.IndexToNode(idx)
        route_indices.append(node_idx)
        idx = sol.Value(mdl.NextVar(idx))
    route_indices.append(mgr.IndexToNode(idx))
    
    optimal_order = []
    for chosen_flat_idx in route_indices[:-1]:
        for pos_enclosure, r_indices in enumerate(enclosure_to_indices_map):
            if chosen_flat_idx in r_indices:
                optimal_order.append(pos_enclosure)
                break
    optimal_order.append(0)
    
    target_nodes = [None] * len(selected_names)
    for pos_enclosure, chosen_flat_idx in zip(optimal_order[:-1], route_indices[:-1]):
        target_nodes[pos_enclosure] = flat_nodes[chosen_flat_idx]
        
    return selected_names, optimal_order, target_nodes, best_distance

In [4]:
def render_route(selected_names, optimal_order, target_nodes):
    """
    Reconstructs the Folium map given the chosen parking subsets.
    """
    # Points of interest mapping
    points_of_interest = {}
    for name, chosen_node in zip(selected_names, target_nodes):
        for pt in candidates_utm[name]:
            if pt_to_node[pt] == chosen_node:
                lon, lat = pt_to_lonlat[pt]
                points_of_interest[name] = [lon, lat]
                break

    # Expand Dijkstra path lines
    full_path_nodes = []
    # Check if target_nodes contains any None
    if None in target_nodes:
        # The model failed to visit some enclosed parts!
        return folium.Map(location=[43.350, -3.852], zoom_start=14)
        
    for i in range(len(optimal_order) - 1):
        u_idx = optimal_order[i]
        v_idx = optimal_order[i + 1]
        node_u = target_nodes[u_idx]
        node_v = target_nodes[v_idx]
        segment = paths_between_nodes[(node_u, node_v)]
        if i > 0:
            segment = segment[1:]
        full_path_nodes.extend(segment)

    route_coordinates = [[G_metros.nodes[node]['lat'], G_metros.nodes[node]['lon']] for node in full_path_nodes]

    # Display Map
    mapa = folium.Map(location=[43.350, -3.852], zoom_start=14, tiles='cartodbpositron')
    folium.PolyLine(route_coordinates, color="#FF5A5F", weight=5, opacity=0.8).add_to(mapa)

    # Map each enclosure to visit number
    name_to_order = {selected_names[idx]: pos + 1 for pos, idx in enumerate(optimal_order[:-1])}

    # Clustered markers
    coords_to_names = defaultdict(list)
    for name, coords in points_of_interest.items():
        coords_to_names[tuple(coords)].append(name)

    for coords, group_names in coords_to_names.items():
        is_entrance = "Main Entrance" in group_names
        bg_color = '#2ca02c' if is_entrance else '#1f77b4'
        nums = sorted(name_to_order[n] for n in group_names)
        num_label = "/".join(str(n) for n in nums)
        popup_text = "<br>".join(f"<b>{n}</b>" for n in group_names)
        folium.Marker(
            location=[coords[1], coords[0]],
            popup=folium.Popup(popup_text, max_width=200),
            icon=folium.DivIcon(
                html=f'''<div style="
                    background-color: {bg_color};
                    color: white;
                    border-radius: 50%;
                    width: 28px;
                    height: 28px;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    font-weight: bold;
                    font-size: 12px;
                    border: 2px solid white;
                    box-shadow: 1px 1px 3px rgba(0,0,0,0.5);
                ">{num_label}</div>''',
                icon_size=(28, 28),
                icon_anchor=(14, 14)
            )
        ).add_to(mapa)
        
    return mapa

## 2. Generate Interactive UI

In [5]:
selectable_names = sorted([n for n in all_names if n != 'Main Entrance'])

# Create a checkbox for each animal
checkboxes = []
for i, name in enumerate(selectable_names):
    # Check the first 5 by default
    cb = widgets.Checkbox(
        value=(i < 5),
        description=name,
        indent=False,
        layout=widgets.Layout(width='auto')
    )
    checkboxes.append(cb)

# Arrange checkboxes in a nice 3-column grid
animal_grid = widgets.GridBox(
    checkboxes, 
    layout=widgets.Layout(
        grid_template_columns="repeat(3, 300px)",
        grid_gap='2px 10px',
        margin='10px 0 20px 0'
    )
)

# New Buttons: Select All / Clear All
btn_select_all = widgets.Button(
    description='Select All',
    button_style='info',
    icon='check-square-o',
    layout=widgets.Layout(width='140px')
)
btn_clear_all = widgets.Button(
    description='Clear All',
    button_style='warning',
    icon='square-o',
    layout=widgets.Layout(width='140px')
)

In [6]:
def on_select_all_click(b):
    for cb in checkboxes:
        cb.value = True

In [7]:
def on_clear_all_click(b):
    for cb in checkboxes:
        cb.value = False

In [ ]:
btn_select_all.on_click(on_select_all_click)
btn_clear_all.on_click(on_clear_all_click)

selection_buttons = widgets.HBox([btn_select_all, btn_clear_all], layout=widgets.Layout(margin='0 0 10px 0'))

btn_generate = widgets.Button(
    description='Generate Optimal Route! 🚙',
    button_style='success',
    icon='map',
    layout=widgets.Layout(width='300px', height='45px', margin='10px 0 20px 0')
)

output_map = widgets.Output(layout=widgets.Layout(width='100%', height='500px', border='1px solid #ccc'))
output_text = widgets.Output()

In [9]:
def on_button_click(b):
    # Gather selected animals from the checkboxes
    selected_animals = [cb.description for cb in checkboxes if cb.value]
    
    with output_text:
        clear_output()
        if not selected_animals:
            print("⚠️ Please select at least one animal to visit!")
            return
        print("Solving MILP path...")
        
    sel, opt_ord, tg_nodes, max_dist = solve_custom_tsp(selected_animals)
    
    with output_text:
        clear_output()
        if opt_ord is None:
            print("Could not find a valid solution!")
            return
            
        print(f"✅ Solution found! Distance: {max_dist / 1000:.2f} km")
        ordered_names = [sel[i] for i in opt_ord[:-1]]
        
    with output_map:
        clear_output(wait=True)
        map_view = render_route(sel, opt_ord, tg_nodes)
        display(map_view)

In [10]:
btn_generate.on_click(on_button_click)

In [11]:
# Display UI Layout
ui_box = widgets.VBox([
    widgets.HTML("<h2>Choose your Cabárceno Safari Trip</h2><p>Select the animals you want to visit and we will calculate the optimal route.</p>"),
    selection_buttons,
    animal_grid,
    btn_generate,
    widgets.HTML("<hr>"),
    output_text,
    output_map
])
display(ui_box)